# 🎫 Day 05a: LLD — Booking System (BookMyShow)

---

## 🎯 What You'll Master Today
- Seat selection with concurrency handling
- Optimistic vs pessimistic locking
- Payment flow & timeout
- Complete class design

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  BOOKING SYSTEM — CORE CHALLENGE: CONCURRENCY                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  The problem: 1000 users trying to book the same seat            ║
║  at the same time. Only ONE should succeed!                      ║
║                                                                   ║
║  Flow:                                                           ║
║  Browse shows → Select seats → [LOCK] → Payment → Confirm      ║
║                                   │                               ║
║                                   └─ timeout? → release seats    ║
║                                                                   ║
║  Class Diagram:                                                  ║
║  ┌────────┐  1:N  ┌──────┐  1:N  ┌────────┐  N:M  ┌──────┐   ║
║  │ Cinema │──────►│Screen│──────►│  Show  │──────►│ Seat │    ║
║  └────────┘       └──────┘       └────────┘       └──────┘    ║
║                                       │                         ║
║                                       │ 1:N                     ║
║                                  ┌────────┐                     ║
║                    User ────────►│Booking │                     ║
║                                  └────────┘                     ║
║                                       │                         ║
║                                  ┌────────┐                     ║
║                                  │Payment │                     ║
║                                  └────────┘                     ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Core Classes
# ============================================================

from enum import Enum
from datetime import datetime, timedelta
import threading
import uuid

class SeatStatus(Enum):
    AVAILABLE = "available"
    LOCKED = "locked"        # temporarily held during payment
    BOOKED = "booked"        # confirmed

class SeatType(Enum):
    REGULAR = "regular"
    PREMIUM = "premium"
    VIP = "vip"

class Seat:
    PRICES = {SeatType.REGULAR: 200, SeatType.PREMIUM: 400, SeatType.VIP: 800}
    
    def __init__(self, row: str, number: int, seat_type: SeatType = SeatType.REGULAR):
        self.row = row
        self.number = number
        self.seat_type = seat_type
        self.status = SeatStatus.AVAILABLE
        self.locked_by = None
        self.lock_expiry = None
    
    @property
    def price(self):
        return self.PRICES[self.seat_type]
    
    @property
    def seat_id(self):
        return f"{self.row}{self.number}"
    
    def is_available(self):
        if self.status == SeatStatus.LOCKED and self.lock_expiry:
            if datetime.now() > self.lock_expiry:
                self.release()  # Lock expired
        return self.status == SeatStatus.AVAILABLE
    
    def lock(self, user_id, timeout_minutes=10):
        self.status = SeatStatus.LOCKED
        self.locked_by = user_id
        self.lock_expiry = datetime.now() + timedelta(minutes=timeout_minutes)
    
    def book(self):
        self.status = SeatStatus.BOOKED
        self.lock_expiry = None
    
    def release(self):
        self.status = SeatStatus.AVAILABLE
        self.locked_by = None
        self.lock_expiry = None
    
    def __repr__(self):
        return f"{self.seat_id}({self.status.value})"


class Show:
    def __init__(self, show_id: str, movie: str, screen: int, showtime: str):
        self.show_id = show_id
        self.movie = movie
        self.screen = screen
        self.showtime = showtime
        self.seats = []
    
    def setup_seats(self, rows="ABCDE", seats_per_row=10):
        for row in rows:
            seat_type = SeatType.VIP if row == 'A' else (
                SeatType.PREMIUM if row == 'B' else SeatType.REGULAR)
            for num in range(1, seats_per_row + 1):
                self.seats.append(Seat(row, num, seat_type))
    
    def available_seats(self):
        return [s for s in self.seats if s.is_available()]
    
    def get_seat(self, seat_id: str):
        for s in self.seats:
            if s.seat_id == seat_id:
                return s
        return None

print("✅ Seat and Show classes defined")

In [ ]:
# ============================================================
# 2. Booking System with Concurrency Handling
# ============================================================

class BookingSystem:
    def __init__(self):
        self.shows = {}      # show_id → Show
        self.bookings = {}   # booking_id → Booking
        self.lock = threading.Lock()
    
    def add_show(self, show: Show):
        self.shows[show.show_id] = show
    
    def select_seats(self, show_id: str, seat_ids: list, user_id: str) -> dict:
        """
        Lock selected seats for the user (optimistic approach with mutex).
        Returns booking_id if successful.
        """
        with self.lock:
            show = self.shows.get(show_id)
            if not show:
                return {"error": "Show not found"}
            
            # Check all seats available
            seats = []
            for sid in seat_ids:
                seat = show.get_seat(sid)
                if not seat or not seat.is_available():
                    return {"error": f"Seat {sid} not available"}
                seats.append(seat)
            
            # Lock all seats atomically
            booking_id = str(uuid.uuid4())[:8]
            total = 0
            for seat in seats:
                seat.lock(user_id)
                total += seat.price
            
            self.bookings[booking_id] = {
                "booking_id": booking_id,
                "user_id": user_id,
                "show_id": show_id,
                "seats": seat_ids,
                "total": total,
                "status": "pending_payment"
            }
            return self.bookings[booking_id]
    
    def confirm_payment(self, booking_id: str) -> dict:
        """Confirm booking after payment."""
        with self.lock:
            booking = self.bookings.get(booking_id)
            if not booking:
                return {"error": "Booking not found"}
            
            show = self.shows[booking["show_id"]]
            for sid in booking["seats"]:
                seat = show.get_seat(sid)
                if seat.locked_by == booking["user_id"]:
                    seat.book()
                else:
                    return {"error": f"Seat {sid} lock expired!"}
            
            booking["status"] = "confirmed"
            return booking
    
    def cancel_booking(self, booking_id: str) -> dict:
        with self.lock:
            booking = self.bookings.get(booking_id)
            if not booking:
                return {"error": "Booking not found"}
            
            show = self.shows[booking["show_id"]]
            for sid in booking["seats"]:
                seat = show.get_seat(sid)
                seat.release()
            
            booking["status"] = "cancelled"
            return booking

# --- Demo ---
system = BookingSystem()
show = Show("S1", "Interstellar", screen=1, showtime="7:00 PM")
show.setup_seats(rows="ABC", seats_per_row=5)
system.add_show(show)

print(f"Available seats: {len(show.available_seats())}")

# User 1 books seats
result = system.select_seats("S1", ["A1", "A2"], "user_001")
print(f"\nUser1 selects A1, A2: {result}")

# User 2 tries same seats (should fail)
result2 = system.select_seats("S1", ["A1", "B3"], "user_002")
print(f"User2 tries A1, B3: {result2}")

# User 2 picks different seats
result3 = system.select_seats("S1", ["B3", "B4"], "user_002")
print(f"User2 picks B3, B4: {result3}")

# Confirm payment for User 1
confirmed = system.confirm_payment(result["booking_id"])
print(f"\nUser1 confirmed: {confirmed}")
print(f"Available seats after booking: {len(show.available_seats())}")

In [ ]:
# ============================================================
# 3. Concurrency Strategies
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════╗
║  CONCURRENCY STRATEGIES FOR BOOKING                          ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  1️⃣ PESSIMISTIC LOCKING (what we implemented)                ║
║     SELECT ... FOR UPDATE  (database-level row lock)          ║
║     • Lock seats when user selects them                      ║
║     • Other users blocked until release                      ║
║     • Pro: No conflict, guaranteed consistency               ║
║     • Con: Holds DB locks, can cause deadlocks               ║
║                                                                ║
║  2️⃣ OPTIMISTIC LOCKING (better for high traffic)            ║
║     • Don't lock. Let everyone select.                       ║
║     • At booking time, check version/timestamp               ║
║     • UPDATE seats SET status='booked' WHERE status='free'   ║
║       AND version = expected_version                          ║
║     • If UPDATE affects 0 rows → someone else booked it!     ║
║     • Pro: No locks held, higher throughput                  ║
║     • Con: User might see "seat unavailable" at payment time ║
║                                                                ║
║  3️⃣ TEMPORARY HOLD (our approach with timeout)              ║
║     • Lock seat for N minutes → user pays → confirm          ║
║     • If timeout → auto-release                              ║
║     • Pro: Best UX (user sees "held for you")                ║
║     • Con: Seats "wasted" during hold if user abandons       ║
║                                                                ║
║  💡 BookMyShow/Ticketmaster: Temporary hold + Redis           ║
║     for distributed locks with auto-expiry (SETEX)           ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Design ticket booking system | Show→Seats, lock on select, timeout, confirm on payment |
| 2 | Handle concurrent bookings? | Pessimistic (SELECT FOR UPDATE) or Optimistic (version check) |
| 3 | What if user abandons? | Temporary hold with TTL. Auto-release seats after timeout |
| 4 | How to scale? | Redis for distributed locks (SETEX). Partition by show/screen |
| 5 | Optimistic vs pessimistic? | Optimistic: no locks, check at commit. Pessimistic: lock upfront |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Core challenge: concurrent access to same seats       │
## │  • Temporary hold + timeout = best UX                    │
## │  • Optimistic locking for high throughput                 │
## │  • Redis SETEX for distributed temporary holds            │
## │  • Atomic seat selection (all or nothing)                 │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Snake Game Design**